In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz
from implicit.nearest_neighbours import CosineRecommender
from collections import defaultdict
from tqdm import tqdm
import os
import gc
from sklearn.preprocessing import normalize

# =============================================================================
# 1. CARGAR Y PREPARAR DATOS
# =============================================================================
print("🚀 Cargando y preparando datos...")
df_pairs_unique = pd.read_pickle("data/df_pairs_unique.pkl")
author2idx = {str(k): int(v) for k, v in np.load("data/author_to_idx.npy", allow_pickle=True).item().items()}
idx2author = {int(k): str(v) for k, v in np.load("data/idx_to_author.npy", allow_pickle=True).item().items()}
n_authors = len(author2idx)

author_counts = df_pairs_unique[['pair_min','pair_max']].stack().value_counts()
eligible_authors = set(str(x) for x in author_counts[author_counts >= 2].index)
df_filtered = df_pairs_unique[
    df_pairs_unique['pair_min'].astype(str).isin(eligible_authors) &
    df_pairs_unique['pair_max'].astype(str).isin(eligible_authors)
].reset_index(drop=True)

author_matrix = load_npz('data/author_concept_matrix.npz').astype(np.float32)
author_matrix = normalize(author_matrix, norm='l2', axis=1, copy=False)
cb_author_ids = np.load('data/cb_author_ids.npy', allow_pickle=True).astype(str)
cb_author_to_idx = {aid: i for i, aid in enumerate(cb_author_ids)}

author_work_counts = np.load('data/cb_author_work_counts.npy').astype(np.float32)

# =============================================================================
# 2. FUNCIONES DE APOYO Y MÉTRICAS
# =============================================================================
def build_csr(df, author2idx, n):
    u_idx = df['pair_min'].astype(str).map(author2idx).values
    v_idx = df['pair_max'].astype(str).map(author2idx).values

    # 1. Extraer los pesos (frecuencia de colaboración)
    # Si la columna se llama 'count', la usamos. Si no, usamos 1 por defecto.
    counts = df['count'].values if 'count' in df.columns else np.ones(len(u_idx))

    # 2. Aplicar la transformación logarítmica log1p
    # Esto evita que autores con 100 papers dominen injustamente sobre los de 5
    weights = np.log1p(counts.astype(np.float32))

    # 3. Duplicar para asegurar simetría (Grafo no dirigido)
    rows = np.concatenate([u_idx, v_idx])
    cols = np.concatenate([v_idx, u_idx])
    data = np.concatenate([weights, weights])

    return csr_matrix((data, (rows, cols)), shape=(n, n))

def build_gt(df):
    gt = defaultdict(set)
    for row in df.itertuples():
        u, v = str(row.pair_min), str(row.pair_max)
        gt[u].add(v); gt[v].add(u)
    return gt

def calculate_metrics(recs, gt, train_pop=None, n_total_authors=None, k=20):
    r_sum, n_sum, count = 0.0, 0.0, 0
    novelty_sum = 0.0
    recommended_items = set()

    for u, rel in gt.items():
        if not rel: continue
        ranked = recs.get(u, [])[:k]
        recommended_items.update(ranked)

        hits = [1 if str(i) in rel else 0 for i in ranked]
        r_sum += sum(hits) / len(rel)

        dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
        n_sum += (dcg / idcg) if idcg > 0 else 0

        if train_pop is not None and n_total_authors is not None:
            for item in ranked:
                pop = train_pop.get(item, 1)
                novelty_sum += -np.log2(pop / n_total_authors)
        count += 1

    if count == 0: return 0, 0, 0, 0

    # Calculamos métricas finales con protecciones contra None
    recall = r_sum / count
    ndcg = n_sum / count
    novelty = (novelty_sum / (count * k)) if (count > 0 and n_total_authors) else 0
    coverage = (len(recommended_items) / n_total_authors) if n_total_authors else 0

    return recall, ndcg, novelty, coverage


def calculate_serendipity(recs_hybrid, recs_cf_base, gt, k=20):
    """
    Serendipity: Mide aciertos en el híbrido que NO estaban en el top-K del CF base.
    """
    serendipity_score = 0.0
    count = 0
    for u, rel in gt.items():
        if not rel: continue
        hyb_top = set(recs_hybrid.get(u, [])[:k])
        cf_top = set(recs_cf_base.get(u, [])[:k])

        # Aciertos inesperados: están en Hyb, están en Rel, pero NO en CF
        unexpected_hits = (hyb_top - cf_top) & rel
        serendipity_score += len(unexpected_hits) / len(rel)
        count += 1
    return serendipity_score / count if count > 0 else 0

# =============================================================================
# 3. RECOMENDADORES
# =============================================================================
def get_itemknn_recs(model, train_matrix, target_authors, topk=50):
    recs = {}
    train_matrix = train_matrix.tocsr()
    for auth in tqdm(target_authors, desc="ItemKNN", leave=False):
        u_idx = author2idx.get(auth)
        if u_idx is None: continue
        ids, scores = model.recommend(userid=u_idx, user_items=train_matrix[u_idx], N=topk)
        recs[auth] = {idx2author[idx]: float(sc) for idx, sc in zip(ids, scores)}
    return recs

def get_cb_recs_batch(target_authors, author_matrix, cb_ids, cb_map, work_counts, C=1.0, topk=50, chunk_size=5000):
    """
    Versión con Bayesian Smoothing incorporado.
    mu: Prior global (media de similitudes).
    C: Constante de confianza.
    work_counts: Array con el número de trabajos por cada autor en cb_ids.
    """
    recs = {}
    target_indices = [cb_map[a] for a in target_authors if a in cb_map]

    mu=0.034

    # Aseguramos que work_counts sea un array de numpy para operaciones vectorizadas
    work_counts = np.array(work_counts, dtype=np.float32)

    for i in tqdm(range(0, len(target_indices), chunk_size), desc="CB Batch (Smoothed)", leave=False):
        batch_idx = target_indices[i:i+chunk_size]

        # 1. Similitud de coseno cruda (vía producto punto)
        sim_matrix = author_matrix[batch_idx].dot(author_matrix.T).toarray()

        for j, auth_idx in enumerate(batch_idx):
            sims = sim_matrix[j]

            # 2. APLICACIÓN DEL BAYESIAN SMOOTHING
            # Ajustamos cada similitud según el volumen de obra del autor comparado
            if C > 0:
                sims = (C * mu + work_counts * sims) / (C + work_counts)

            # Excluir self-recommendation
            sims[auth_idx] = -1.0

            # 3. Obtener Top-K
            top_i = np.argpartition(-sims, topk)[:topk]
            top_i = top_i[np.argsort(-sims[top_i])]

            recs[cb_ids[auth_idx]] = {cb_ids[idx]: float(sims[idx]) for idx in top_i}

    return recs

# =============================================================================
# 4. SPLIT Y TUNING (VALIDACIÓN)
# =============================================================================
def triple_loo_split(df, seed=42):
    rng = np.random.default_rng(seed)
    adj = defaultdict(list)
    for idx, row in enumerate(df.itertuples()):
        adj[str(row.pair_min)].append(idx); adj[str(row.pair_max)].append(idx)
    test_idx, val_idx, assigned = set(), set(), set()
    for author in adj:
        indices = [i for i in adj[author] if i not in assigned]
        if len(indices) >= 3:
            rng.shuffle(indices); test_idx.add(indices[0]); val_idx.add(indices[1])
            assigned.update([indices[0], indices[1]])
        elif len(indices) == 2:
            t = rng.choice(indices); test_idx.add(t); assigned.add(t)
    train_idx = np.setdiff1d(np.arange(len(df)), list(test_idx | val_idx))
    return df.iloc[train_idx], df.iloc[list(val_idx)], df.iloc[list(test_idx)]

df_train, df_val, df_test = triple_loo_split(df_filtered)
X_train = build_csr(df_train, author2idx, n_authors)
gt_val = build_gt(df_val)
val_authors = list(gt_val.keys())

#print("\n--- Entrenando y Generando Bases para Tuning ---")
#knn_val_model = CosineRecommender(K=200); knn_val_model.fit(X_train, show_progress=False)
#raw_knn_val = get_itemknn_recs(knn_val_model, X_train, val_authors)
#raw_cb_val = get_cb_recs_batch(val_authors, author_matrix, cb_author_ids, cb_author_to_idx)

#print("\n--- Tuning de Alpha en Validación ---")
#best_alpha, best_r = 0, 0
#for a in np.linspace(0, 1, 11):
#    recs_h = {}
#    for auth in val_authors:
#        k_d, c_d = raw_knn_val.get(auth, {}), raw_cb_val.get(auth, {})
#        all_cands = set(k_d.keys()) | set(c_d.keys())
#        # Fusión con Alpha
#        scores = sorted([(c, a*c_d.get(c,0) + (1-a)*k_d.get(c,0)) for c in all_cands], key=lambda x:x[1], reverse=True)
#        recs_h[auth] = [x[0] for x in scores[:20]]

#    # AQUÍ ESTABA EL ERROR: Añadimos n_total_authors=n_authors
#    r, n, _, _ = calculate_metrics(recs_h, gt_val, n_total_authors=n_authors)
#    print(f"Alpha: {a:.1f} | Recall: {r:.4f} | NDCG: {n:.4f}")
#    if r > best_r: best_r, best_alpha = r, a

#print(f"\n🏆 MEJOR ALPHA ENCONTRADO: {best_alpha}")

# =============================================================================
# 5. EVALUACIÓN FINAL (TEST)
# =============================================================================
print("\n" + "="*50 + "\nEVALUACIÓN FINAL EN TEST\n" + "="*50)
df_final_train = pd.concat([df_train, df_val])
X_final_train = build_csr(df_final_train, author2idx, n_authors)
gt_test = build_gt(df_test)
test_authors = list(gt_test.keys())
train_pop = df_final_train[['pair_min','pair_max']].stack().astype(str).value_counts().to_dict()

final_knn = CosineRecommender(K=200); final_knn.fit(X_final_train, show_progress=False)
final_raw_knn = get_itemknn_recs(final_knn, X_final_train, test_authors)
final_raw_cb = get_cb_recs_batch(test_authors, author_matrix, cb_author_ids, cb_author_to_idx)

best_alpha = 0.2 # Fix para ahorrar partes
recs_final_h = {}
for auth in test_authors:
    k_d, c_d = final_raw_knn.get(auth, {}), final_raw_cb.get(auth, {})
    all_cands = set(k_d.keys()) | set(c_d.keys())
    scores = sorted([(c, best_alpha*c_d.get(c,0) + (1-best_alpha)*k_d.get(c,0)) for c in all_cands], key=lambda x:x[1], reverse=True)
    recs_final_h[auth] = [x[0] for x in scores[:20]]

# Resultados Finales
r, n, nov, cov = calculate_metrics(recs_final_h, gt_test, train_pop, n_authors)
# Convertir formato de recs de ItemKNN para serendipity (de dict a lista)
recs_cf_only = {u: list(d.keys()) for u, d in final_raw_knn.items()}
seren = calculate_serendipity(recs_final_h, recs_cf_only, gt_test)

print(f"\n🎯 RESULTADOS HÍBRIDOS FINALES (Alpha={best_alpha}):")
print(f"Recall@20:   {r:.4f}")
print(f"NDCG@20:     {n:.4f}")
print(f"Novelty:     {nov:.4f}")
print(f"Coverage:    {cov*100:.2f}%")
print(f"Serendipity: {seren:.4f}")